# Data Processing - LittleSteps Assignment

This notebook contains the complete **data preparation and feature engineering workflow** for the LittleSteps at-home healthcare assignment.

The goal of this notebook is to convert the raw visit-level dataset into a clean and analysis-ready dataset that can be used in the data analysis notebook.

The processing workflow covers:

1. Loading the raw dataset
2. Initial data exploration
3. Duplicate handling
4. Missing value treatment
5. Date/time standardisation
6. Categorical variable cleaning
7. Visit duration calculation
8. Travel duration estimation
9. Visit duration outlier treatment
10. Nurse note cleaning and keyword-based feature extraction
11. Exporting the final processed dataset

## 1. Import Required Libraries

The following libraries are used for data loading, data cleaning, feature engineering, and basic quality checks.

In [1]:
# Core data handling libraries
import pandas as pd
import numpy as np

# File path handling
from pathlib import Path

# Text cleaning and keyword extraction
import re

# Display settings for better notebook readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 2. Load the Raw Dataset

The raw dataset is expected to be stored inside the `Datasets` folder.

The code below uses a relative path first, which makes the notebook easier to run after cloning the GitHub repository. If the relative path does not work in a different local setup, update the `raw_data_path` manually.

In [2]:
# Define possible raw data file paths.
# The first option works when this notebook is inside the "Data Processing" folder
# and the dataset is inside the "Datasets" folder at the repository root.
possible_paths = [
    Path('../Datasets/visits_raw.csv'),
    Path('Datasets/visits_raw.csv'),
    Path('visits_raw.csv')
]

# Select the first path that exists
raw_data_path = next((path for path in possible_paths if path.exists()), None)

# Raise a clear error if the file cannot be found
if raw_data_path is None:
    raise FileNotFoundError(
        "visits_raw.csv was not found. Please update raw_data_path to match the file location on your system."
    )

# Load the raw dataset
visits_raw = pd.read_csv(raw_data_path)

# Create a working copy so that the original raw dataframe remains unchanged
visits = visits_raw.copy()

print(f"Raw dataset loaded successfully from: {raw_data_path}")
print(f"Raw dataset shape: {visits.shape[0]} rows and {visits.shape[1]} columns")

Raw dataset loaded successfully from: ..\Datasets\visits_raw.csv
Raw dataset shape: 1000 rows and 8 columns


## 3. Initial Data Exploration

Before cleaning the data, the structure of the dataset is reviewed to understand:

- Number of rows and columns
- Column names
- Data types
- Missing values
- Duplicate visit IDs
- Unique values in categorical columns

In [3]:
# Display the first few rows of the raw dataset
visits.head()

,visit_id,patient_id,nurse_id,visit_start_time,visit_end_time,service_type,visit_location,nurse_notes
0,6a4a715f-94c5-4b5d-8f51-51f07f298fb6,P59581,N1565,2025-09-02 00:08:24,2025-09-02 00:55:24,Wound Care,East,The nurse changed the wound dressing. The pati...
1,27bfc4de-9dd2-4b6b-b05f-f9560a3fd00f,P66547,N5698,"August 24, 2025 10:42PM","August 25, 2025 12:12AM",General Check-up,East,The nurse reviewed the oxygen level. The patie...
2,9c2d0018-8f20-4789-a383-906d49042fbc,P47919,N3012,2025-09-02 11:07:33,2025/09/02 11:56,Physical Therapy,East,Review is required. The nurse assisted the mob...
3,bbedc08d-0ada-4c8f-bac4-3008ac279a01,P18359,N8991,2025-09-15 21:24:42,2025-09-15 22:12:42,General Check-up,North,Follow-up is required. The nurse reviewed the ...
4,155c3765-22d5-41cc-95f5-a94ef19748ae,P45602,N7948,2025-08-31 07:19:11,2025-08-31 07:47:11,Pyhcisal Therapy,North,ASAP is required. The nurse observed the patie...


In [4]:
# Check the number of rows and columns
visits.shape

(1000, 8)

In [5]:
# Check column names and data types
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   visit_id          1000 non-null   object
 1   patient_id        1000 non-null   object
 2   nurse_id          1000 non-null   object
 3   visit_start_time  1000 non-null   object
 4   visit_end_time    900 non-null    object
 5   service_type      1000 non-null   object
 6   visit_location    1000 non-null   object
 7   nurse_notes       900 non-null    object
dtypes: object(8)
memory usage: 62.6+ KB


In [6]:
# Summary statistics for all columns, including categorical variables
visits.describe(include='all')

,visit_id,patient_id,nurse_id,visit_start_time,visit_end_time,service_type,visit_location,nurse_notes
count,1000,1000,1000,1000,900,1000,1000,900
unique,970,430,100,1000,900,8,8,678
top,25e18a0c-9cf3-427c-bedf-0a27e8060396,P29486,N2273,2025-09-02 00:08:24,2025-09-02 00:55:24,Medication Administration,North,The nurse provided the IV line. The patient is...
freq,2,8,19,1,1,243,244,6


In [7]:
# Check missing values in each column
missing_summary = visits.isna().sum().reset_index()
missing_summary.columns = ['column_name', 'missing_count']
missing_summary['missing_percentage'] = (missing_summary['missing_count'] / len(visits) * 100).round(2)
missing_summary

,column_name,missing_count,missing_percentage
0,visit_id,0,0.0
1,patient_id,0,0.0
2,nurse_id,0,0.0
3,visit_start_time,0,0.0
4,visit_end_time,100,10.0
5,service_type,0,0.0
6,visit_location,0,0.0
7,nurse_notes,100,10.0


In [8]:
# Check duplicate records based on visit_id, as required in the assignment
duplicate_visit_count = visits.duplicated(subset='visit_id').sum()
print(f"Number of duplicated visit_id records: {duplicate_visit_count}")

Number of duplicated visit_id records: 30


In [9]:
# Review unique values in categorical columns before cleaning
for column in ['service_type', 'visit_location']:
    print(f"\nUnique values in {column}:")
    print(visits[column].value_counts(dropna=False))


Unique values in service_type:
service_type
Medication Administration    243
Wound Care                   239
General Check-up             221
Physical Therapy             217
Pyhcisal Therapy              24
Medicatn Adminstratino        21
Wound Cae                     20
General Chek-up               15
Name: count, dtype: int64

Unique values in visit_location:
visit_location
North     244
West      244
South     244
East      218
Wsst       15
Easst      14
Notrh      11
Soutth     10
Name: count, dtype: int64


## 4. Basic Text Standardisation

Before applying specific cleaning rules, all text columns are stripped of unnecessary leading and trailing spaces.

This step helps avoid issues where values such as `East` and ` East ` are treated as different categories.

In [10]:
# Identify text columns
text_columns = visits.select_dtypes(include=['object']).columns

# Strip leading and trailing spaces from all text columns
for column in text_columns:
    visits[column] = visits[column].astype('string').str.strip()

## 5. Remove Duplicate Visit IDs

The assignment specifically requires duplicate records to be removed based on `visit_id`.

Some duplicated `visit_id` rows may not be exact duplicates. Therefore, instead of simply keeping the first row, a simple data quality score is created and the most complete row is retained for each duplicated `visit_id`.

Rows with more complete key information are prioritised.

In [11]:
# Store the number of rows before duplicate removal
rows_before_deduplication = len(visits)

# Columns considered important for retaining the most complete duplicate record
key_columns = [
    'patient_id',
    'nurse_id',
    'visit_start_time',
    'visit_end_time',
    'service_type',
    'visit_location',
    'nurse_notes'
]

# Create a quality score based on the number of non-missing key fields
visits['_data_quality_score'] = visits[key_columns].notna().sum(axis=1)

# Sort by visit_id and quality score so that the most complete record is kept
visits = (
    visits
    .sort_values(by=['visit_id', '_data_quality_score'], ascending=[True, False])
    .drop_duplicates(subset='visit_id', keep='first')
    .drop(columns=['_data_quality_score'])
    .reset_index(drop=True)
)

rows_after_deduplication = len(visits)
removed_duplicates = rows_before_deduplication - rows_after_deduplication

print(f"Rows before duplicate removal: {rows_before_deduplication}")
print(f"Rows after duplicate removal: {rows_after_deduplication}")
print(f"Duplicate rows removed: {removed_duplicates}")

Rows before duplicate removal: 1000
Rows after duplicate removal: 970
Duplicate rows removed: 30


## 6. Standardise Date and Time Columns

The raw dataset contains date and time values in different formats. These are converted into standard pandas datetime format.

The two key timestamp columns are:

- `visit_start_time`
- `visit_end_time`

Rows with missing or invalid start/end timestamps cannot be used to calculate visit duration accurately. These rows are removed from the processed dataset after being counted and documented.

In [12]:
def parse_mixed_datetime(series):
    """
    Converts a pandas Series containing mixed datetime formats into pandas datetime format.
    Uses pandas' mixed-format parser where available, with a fallback for older pandas versions.
    """
    try:
        return pd.to_datetime(series, errors='coerce', format='mixed')
    except TypeError:
        return pd.to_datetime(series, errors='coerce')

# Convert timestamp columns to datetime format
visits['visit_start_time'] = parse_mixed_datetime(visits['visit_start_time'])
visits['visit_end_time'] = parse_mixed_datetime(visits['visit_end_time'])

# Check whether any values could not be parsed
datetime_missing_summary = visits[['visit_start_time', 'visit_end_time']].isna().sum()
datetime_missing_summary

visit_start_time     0
visit_end_time      92
dtype: int64

In [13]:
# Remove rows where visit start or end time is missing after conversion
# Justification: visit duration cannot be calculated without both timestamps.
rows_before_timestamp_filter = len(visits)

visits = visits.dropna(subset=['visit_start_time', 'visit_end_time']).reset_index(drop=True)

rows_after_timestamp_filter = len(visits)
removed_timestamp_rows = rows_before_timestamp_filter - rows_after_timestamp_filter

print(f"Rows before removing missing timestamp records: {rows_before_timestamp_filter}")
print(f"Rows after removing missing timestamp records: {rows_after_timestamp_filter}")
print(f"Rows removed due to missing start/end timestamp: {removed_timestamp_rows}")

Rows before removing missing timestamp records: 970
Rows after removing missing timestamp records: 878
Rows removed due to missing start/end timestamp: 92


## 7. Clean and Standardise Categorical Variables

The raw dataset contains spelling inconsistencies in categorical columns.

For example:

- `Pyhcisal Therapy` should be `Physical Therapy`
- `Wound Cae` should be `Wound Care`
- `Wsst` should be `West`

The following code standardises `service_type` and `visit_location` into consistent categories.

In [14]:
# Standardisation map for service_type
service_type_map = {
    'physical therapy': 'Physical Therapy',
    'pyhcisal therapy': 'Physical Therapy',
    'medication administration': 'Medication Administration',
    'medicatn adminstratino': 'Medication Administration',
    'wound care': 'Wound Care',
    'wound cae': 'Wound Care',
    'general check-up': 'General Check-up',
    'general chek-up': 'General Check-up'
}

# Standardisation map for visit_location
location_map = {
    'north': 'North',
    'notrh': 'North',
    'south': 'South',
    'soutth': 'South',
    'east': 'East',
    'easst': 'East',
    'west': 'West',
    'wsst': 'West'
}

# Apply cleaning maps
visits['service_type'] = visits['service_type'].str.lower().map(service_type_map)
visits['visit_location'] = visits['visit_location'].str.lower().map(location_map)

# Check if any category values remain unmapped
print("Missing service_type values after mapping:", visits['service_type'].isna().sum())
print("Missing visit_location values after mapping:", visits['visit_location'].isna().sum())

print("\nCleaned service_type values:")
print(visits['service_type'].value_counts())

print("\nCleaned visit_location values:")
print(visits['visit_location'].value_counts())

Missing service_type values after mapping: 0
Missing visit_location values after mapping: 0

Cleaned service_type values:
service_type
Medication Administration    236
Wound Care                   220
Physical Therapy             213
General Check-up             209
Name: count, dtype: int64

Cleaned visit_location values:
visit_location
West     229
North    228
East     212
South    209
Name: count, dtype: int64


## 8. Calculate Visit Duration

A new column named `visit_duration_minutes` is created using the difference between `visit_end_time` and `visit_start_time`.

The formula is:

`visit_duration_minutes = visit_end_time - visit_start_time`

The duration is converted into minutes for easier interpretation.

In [15]:
# Calculate visit duration in minutes
visits['visit_duration_original_minutes'] = (
    visits['visit_end_time'] - visits['visit_start_time']
).dt.total_seconds() / 60

# Check for impossible duration values such as negative or zero durations
invalid_duration_rows = visits[visits['visit_duration_original_minutes'] <= 0]
print(f"Number of records with zero or negative visit duration: {len(invalid_duration_rows)}")

# Remove invalid duration rows if any exist
# Justification: negative or zero visit durations are not operationally meaningful for this analysis.
visits = visits[visits['visit_duration_original_minutes'] > 0].reset_index(drop=True)

# Display duration summary before outlier treatment
visits['visit_duration_original_minutes'].describe()

Number of records with zero or negative visit duration: 0


count     878.000000
mean       85.942995
std       149.610188
min         0.383333
25%        34.016667
50%        62.983333
75%        94.683333
max      1432.883333
Name: visit_duration_original_minutes, dtype: float64

## 9. Address Visit Duration Outliers

The assignment specifically asks for outliers in visit durations to be addressed.

The Interquartile Range method is used:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 - Q1
- Lower bound = Q1 - 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Instead of deleting outlier records, visit durations are capped at the upper and lower bounds. This keeps the observations in the dataset while reducing the influence of extreme values on averages.

A separate flag column named `is_visit_duration_outlier` is also created to preserve information about which visits were originally outliers.

In [16]:
# Calculate IQR bounds for visit duration
q1 = visits['visit_duration_original_minutes'].quantile(0.25)
q3 = visits['visit_duration_original_minutes'].quantile(0.75)
iqr = q3 - q1

lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr

# Flag original outlier records
visits['is_visit_duration_outlier'] = (
    (visits['visit_duration_original_minutes'] < lower_bound) |
    (visits['visit_duration_original_minutes'] > upper_bound)
)

# Create the final cleaned visit_duration_minutes column by capping outlier values
visits['visit_duration_minutes'] = visits['visit_duration_original_minutes'].clip(
    lower=lower_bound,
    upper=upper_bound
)

print(f"Visit duration lower bound: {lower_bound:.2f} minutes")
print(f"Visit duration upper bound: {upper_bound:.2f} minutes")
print(f"Number of visit duration outliers flagged: {visits['is_visit_duration_outlier'].sum()}")

# Compare duration statistics before and after capping
pd.DataFrame({
    'Before Outlier Treatment': visits['visit_duration_original_minutes'].describe(),
    'After Outlier Treatment': visits['visit_duration_minutes'].describe()
})

Visit duration lower bound: 0.00 minutes
Visit duration upper bound: 185.68 minutes
Number of visit duration outliers flagged: 27


,Before Outlier Treatment,After Outlier Treatment
count,878.000000,878.000000
mean,85.942995,66.179784
std,149.610188,39.328931
min,0.383333,0.383333
25%,34.016667,34.016667
50%,62.983333,62.983333
75%,94.683333,94.683333
max,1432.883333,185.683333


## 10. Estimate Travel Duration (Actually Transition Time between Consecutive Appointments or Shifts)

The raw dataset does not directly provide a travel duration column. Therefore, travel duration is estimated using the time gap between a nurse's previous visit ending and their next visit starting.

For each nurse:

`travel_duration_minutes = current visit_start_time - previous visit_end_time`

This should be interpreted as an estimated travel or transition time between consecutive visits. It may include waiting time or scheduling gaps, since actual GPS travel time is not available in the dataset.

The first recorded visit for each nurse will not have a previous visit, so its travel duration is left as missing.

Travel duration is calculated only on the basis of difference between 2 appointments, it is possible that there is a difference of a day between 2 appointments. We could think of providing a shift timing feature to calculate more closer travel time but considering how medical personnel have varying shifts, the travel duration would be incorrect.

In [17]:
# Sort visits by nurse and visit start time before calculating previous visit end time
visits = visits.sort_values(by=['nurse_id', 'visit_start_time']).reset_index(drop=True)

# Get previous visit end time and previous location for each nurse
visits['previous_visit_end_time'] = visits.groupby('nurse_id')['visit_end_time'].shift(1)
visits['previous_visit_location'] = visits.groupby('nurse_id')['visit_location'].shift(1)

# Calculate travel/transition duration in minutes
visits['travel_duration_minutes'] = (
    visits['visit_start_time'] - visits['previous_visit_end_time']
).dt.total_seconds() / 60

# If the calculated gap is negative, it means visits overlap or records are inconsistent.
# These values are set to missing because they cannot represent valid travel time.
visits.loc[visits['travel_duration_minutes'] < 0, 'travel_duration_minutes'] = np.nan

# Optional location movement flag
visits['changed_location_from_previous_visit'] = np.where(
    visits['previous_visit_location'].isna(),
    np.nan,
    visits['visit_location'] != visits['previous_visit_location']
)

# Display travel duration summary
visits['travel_duration_minutes'].describe()

count      762.000000
mean      4237.653543
std       3984.283096
min         12.116667
25%       1358.675000
50%       2903.041667
75%       5841.412500
max      24975.100000
Name: travel_duration_minutes, dtype: float64

## 11. Clean Nurse Notes and Extract Useful Information

The `nurse_notes` column contains unstructured text. Some notes are missing and some contain unnecessary symbols.

The following steps are applied:

1. Missing notes are filled with `No notes provided`
2. Special characters are removed
3. Text is converted into a cleaner lowercase version
4. Keyword-based flags are created to capture common patient outcomes or needs

These extracted features can be used later to understand whether certain patient conditions are associated with longer visits.

In [18]:
def clean_note_text(note):
    """
    Cleans nurse note text by removing unnecessary symbols and extra spaces.
    """
    if pd.isna(note):
        return 'no notes provided'
    note = str(note).lower()
    note = re.sub(r'[^a-z0-9\s\-]', ' ', note)
    note = re.sub(r'\s+', ' ', note).strip()
    return note

# Fill missing nurse notes and create cleaned note text
visits['nurse_notes'] = visits['nurse_notes'].fillna('No notes provided')
visits['nurse_notes_clean'] = visits['nurse_notes'].apply(clean_note_text)

# Keyword flags based on common patient conditions, outcomes, and needs
keyword_patterns = {
    'note_mentions_urgent': r'urgent|asap|critical',
    'note_mentions_follow_up': r'follow\s*-?\s*up|review|required|ongoing assessment',
    'note_mentions_pain': r'pain',
    'note_mentions_weakness_or_dizziness': r'weak|dizzy|restless',
    'note_mentions_stable_or_improving': r'stable|improving|comfortable',
    'note_mentions_medication': r'medication|injection|iv line',
    'note_mentions_wound_care': r'wound|dressing|bandage|incision',
    'note_mentions_mobility_support': r'mobility|range of motion|exercise'
}

# Create one binary column for each keyword group
for column_name, pattern in keyword_patterns.items():
    visits[column_name] = visits['nurse_notes_clean'].str.contains(pattern, regex=True, na=False).astype(int)

In [19]:
def classify_note_outcome(note):
    """
    Creates a simple outcome/need category from nurse notes.
    Priority is assigned so that more operationally important conditions are labelled first.
    """
    if 'no notes provided' in note:
        return 'No Notes Provided'
    elif re.search(r'critical|urgent|asap', note):
        return 'Urgent / Critical Attention'
    elif re.search(r'follow\s*-?\s*up|review|required|ongoing assessment', note):
        return 'Follow-up / Review Required'
    elif re.search(r'pain|weak|dizzy|restless', note):
        return 'Patient Discomfort / Weakness'
    elif re.search(r'stable|improving|comfortable', note):
        return 'Stable / Improving'
    else:
        return 'General Notes'

# Create a higher-level note outcome category
visits['note_outcome_category'] = visits['nurse_notes_clean'].apply(classify_note_outcome)

# Review extracted note categories
visits['note_outcome_category'].value_counts()

note_outcome_category
Follow-up / Review Required    403
Stable / Improving             229
Urgent / Critical Attention    158
No Notes Provided               88
Name: count, dtype: int64

## 12. Final Data Quality Checks

Before exporting the processed dataset, final checks are performed to confirm:

- No duplicate visit IDs remain
- Key duration columns have been created
- Categorical values are standardised
- The final dataset has a clean structure for analysis

In [20]:
# Check final shape
print(f"Final processed dataset shape: {visits.shape[0]} rows and {visits.shape[1]} columns")

# Confirm no duplicate visit IDs remain
print(f"Remaining duplicated visit_id records: {visits.duplicated(subset='visit_id').sum()}")

# Check missing values in important engineered columns
important_columns = [
    'visit_id',
    'patient_id',
    'nurse_id',
    'visit_start_time',
    'visit_end_time',
    'service_type',
    'visit_location',
    'visit_duration_minutes',
    'travel_duration_minutes',
    'note_outcome_category'
]

visits[important_columns].isna().sum()

Final processed dataset shape: 878 rows and 25 columns
Remaining duplicated visit_id records: 0


visit_id                     0
patient_id                   0
nurse_id                     0
visit_start_time             0
visit_end_time               0
service_type                 0
visit_location               0
visit_duration_minutes       0
travel_duration_minutes    116
note_outcome_category        0
dtype: int64

In [21]:
# Review final dataset preview
visits.head()

,visit_id,patient_id,nurse_id,visit_start_time,visit_end_time,service_type,visit_location,nurse_notes,visit_duration_original_minutes,is_visit_duration_outlier,visit_duration_minutes,previous_visit_end_time,previous_visit_location,travel_duration_minutes,changed_location_from_previous_visit,nurse_notes_clean,note_mentions_urgent,note_mentions_follow_up,note_mentions_pain,note_mentions_weakness_or_dizziness,note_mentions_stable_or_improving,note_mentions_medication,note_mentions_wound_care,note_mentions_mobility_support,note_outcome_category
0,a7c0d48c-26d5-47f4-a5c6-97b93fb08c0b,P60703,N1066,2025-08-20 20:30:41,2025-08-20 21:42:41,Medication Administration,East,No notes provided,72.0,False,72.0,NaT,NaN,NaN,NaN,no notes provided,0,0,0,0,0,0,0,0,No Notes Provided
1,ec09fa84-50de-4670-91f1-83a2b290f1cb,P89830,N1066,2025-08-26 06:00:49,2025-08-26 06:01:49,Medication Administration,North,Follow-up is required. The nurse delivered the...,1.0,False,1.0,2025-08-20 21:42:41,East,7698.133333,1.0,follow-up is required the nurse delivered the ...,0,1,0,0,1,0,0,0,Follow-up / Review Required
2,b64afacd-cb51-40c3-bb57-e054ae3ea082,P76027,N1066,2025-09-06 17:45:30,2025-09-06 18:31:30,Physical Therapy,South,Monitoring is required. The nurse encouraged t...,46.0,False,46.0,2025-08-26 06:01:49,North,16543.683333,1.0,monitoring is required the nurse encouraged th...,0,1,0,1,0,0,0,0,Follow-up / Review Required
3,4d217cf0-d5bf-4c54-9072-9ac0ada81d1f,P35198,N1066,2025-09-06 20:23:00,2025-09-06 21:02:00,General Check-up,East,The nurse checked the oxygen level. The patien...,39.0,False,39.0,2025-09-06 18:31:30,South,111.500000,1.0,the nurse checked the oxygen level the patient...,0,0,0,0,1,0,0,0,Stable / Improving
4,1055d02d-3484-430b-bfb2-b59fc9431683,P60246,N1066,2025-09-07 19:42:23,2025-09-07 20:53:23,Wound Care,South,The nurse changed the bandage. The patient is ...,71.0,False,71.0,2025-09-06 21:02:00,East,1360.383333,1.0,the nurse changed the bandage the patient is i...,1,1,1,0,0,0,1,0,Urgent / Critical Attention


In [22]:
# Review final column list
visits.columns.tolist()

['visit_id',
 'patient_id',
 'nurse_id',
 'visit_start_time',
 'visit_end_time',
 'service_type',
 'visit_location',
 'nurse_notes',
 'visit_duration_original_minutes',
 'is_visit_duration_outlier',
 'visit_duration_minutes',
 'previous_visit_end_time',
 'previous_visit_location',
 'travel_duration_minutes',
 'changed_location_from_previous_visit',
 'nurse_notes_clean',
 'note_mentions_urgent',
 'note_mentions_follow_up',
 'note_mentions_pain',
 'note_mentions_weakness_or_dizziness',
 'note_mentions_stable_or_improving',
 'note_mentions_medication',
 'note_mentions_wound_care',
 'note_mentions_mobility_support',
 'note_outcome_category']

## 13. Export the Processed Dataset

The cleaned and feature-engineered dataset is exported as:

`visits_processed.csv`

This file will be used as the input dataset for the Data Analysis notebook.

In [23]:
# Define output path for the processed dataset
# This saves the file into the Datasets folder when the repository structure is maintained.
output_path = Path('../Datasets/visits_processed.csv')

# If the Datasets folder does not exist in the current local setup, save in the current folder instead.
if not output_path.parent.exists():
    output_path = Path('visits_processed.csv')

# Export processed dataset
visits.to_csv(output_path, index=False)

print(f"Processed dataset successfully saved to: {output_path}")

Processed dataset successfully saved to: ..\Datasets\visits_processed.csv


## 14. Summary of Data Processing Decisions

The following data processing decisions were made:

1. Duplicate records were removed based on `visit_id`, retaining the most complete record.
2. `visit_start_time` and `visit_end_time` were converted into standard datetime format.
3. Rows with missing start or end timestamps were removed because visit duration could not be calculated without them.
4. Inconsistent `service_type` and `visit_location` values were standardised.
5. `visit_duration_minutes` was calculated from visit start and end timestamps.
6. Visit duration outliers were identified using the IQR method and capped rather than removed.
7. `travel_duration_minutes` was estimated as the time gap between a nurse's previous visit ending and next visit starting.
8. Missing nurse notes were filled, cleaned, and converted into useful keyword-based features.
9. The final processed dataset was exported for analysis.